<h1 align='center'> ELCardio Competition Ensembling Cascade Method </h1>

## Setup & Imports

In [7]:
pip install iterative-stratification

^C
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
import nltk
import unicodedata
import itertools
import torch
import torch.nn.functional as F

from tqdm import tqdm
from torch import nn
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import precision_recall_fscore_support, classification_report, f1_score
from tqdm import trange
from nltk import tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from dataclasses import dataclass
from sklearn.preprocessing import MultiLabelBinarizer
from typing import List, Dict, Optional, Iterator, Any

AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Data Loading & Parsing

### Dataclasses

In [ ]:
@dataclass
class Mention:
    """
    Represents a mention-level annotation within the clinical text.
    
    Attributes:
        start (int): The starting character offset in the document.
        end (int): The ending character offset in the document.
        code (str): The assigned ICD-10 diagnostic code.
        text (str): The actual text span (e.g., "συγκοπτικό επεισόδιο").
    """
    start: int
    end: int
    code: str
    mention: str


@dataclass
class ClinicalDocument:
    """
    Represents a complete clinical document (discharge letter) from the dataset.
    
    Attributes:
        patient_id (Any): Unique identifier for the patient/record.
        text (str): The raw unstructured clinical narrative.
        document_labels (List[List[str]]): A list of lists containing valid ICD-10 codes 
            for the document. The system must predict at least one code from each inner list.
        mentions (Optional[List[Mention]]): A list of mention-level annotations, if available.
    """
    patient_id: Any
    text: str
    document_labels: List[List[str]]
    mentions: Optional[List[Mention]] = None

### Read from jsonl

In [ ]:
filepath = "/kaggle/input/datasets/dimitriospakakis/elcardio-dataset/trainELCardioCC2026.jsonl"
all_documents= []
with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
       json_line = json.loads(line)

       # Load the basic data
       patient_id = json_line['patient_id']
       text = json_line['text']
       document_level_annotations = json_line['document_level_annotations']

       # Load the mentions
       raw_mentions = json_line['mention_level_annotations'] 

       final_mentions = None
      
       # If the mentions list is not empty then we create Mention objects 
       if raw_mentions:
         final_mentions = [Mention(m['start'], m['end'], m['code'], m['mention']) for m in raw_mentions]
       
       all_documents.append(ClinicalDocument(patient_id,text,document_level_annotations,final_mentions))
        
print(len(all_documents))


## Dataset Split

We will split the dataset to 90% train and 5% validation and 5% test. We will handle the codes that are rare. If a code has only 1 document, it will be in the training set. If there are more than 2 documents then there will be an uniform distribution in both test, train and validation sets

We can not convert the y_labels list to numpy array or tensor because every document has a different number of codes, but the 2nd dimension must be the same for all the samples. So, to improve the performance of the training and to solve this issue, we will convert the codes with multi-Hot Encoding. Each label list will be converted to a list where the true codes will be converted to 1 and the false to 0.

In [10]:
X_texts = []
y_labels_raw = []
y_labels = []
for doc in all_documents:
    X_texts.append(doc.text)
    
    current_labels = []
    for label_list in doc.document_labels:
        for label in label_list:
            current_labels.append(label)
    y_labels.append(current_labels.copy())
    # Add extra flag to stratify the set using the labels and the mentions
    if doc.mentions and len(doc.mentions) > 0:
        current_labels.append("HAS_MENTION")
        
    y_labels_raw.append(current_labels)

X_texts = np.array(X_texts)

# Binarization
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y_labels_raw)

mlb2 = MultiLabelBinarizer()
y_labels = mlb2.fit_transform(y_labels)

# 3. Stratified Split (90% Train, 5% Val, 5% Test)
X_indices = np.arange(len(all_documents))

# --- Split: 90% Train / 10% Temp ---
msss_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, temp_idx = next(msss_1.split(X_indices, y_encoded))

# --- Split: 5% Val / 5% Test ---
msss_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_local_idx, test_local_idx = next(msss_2.split(temp_idx, y_encoded[temp_idx]))

val_idx = temp_idx[val_local_idx]
test_idx = temp_idx[test_local_idx]

train_docs = [all_documents[i] for i in train_idx]
val_docs = [all_documents[i] for i in val_idx]
test_docs = [all_documents[i] for i in test_idx]

y_train = y_labels[train_idx]
y_val = y_labels[val_idx]
y_test = y_labels[test_idx]

string_labels = mlb2.classes_
# Εκτυπώσεις
print(f"y_labels shape: {y_labels.shape}")
print(f"Example y_labels[0]: {y_labels[0]}")
print(f"Labels document 0: {string_labels[y_labels[0] == 1]}\n")

print(f"Total documents: {len(all_documents)}")
print("-" * 30)
print(f"Train Set:      {len(train_docs)}")
print(f"Validation Set: {len(val_docs)}")
print(f"Test Set:       {len(test_docs)}")

y_labels shape: (2500, 110)
Example y_labels[0]: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
Labels document 0: ['I10' 'I11' 'I44' 'R55' 'Z95']

Total documents: 2500
------------------------------
Train Set:      2238
Validation Set: 134
Test Set:       128


In [11]:
tokenizer = AutoTokenizer.from_pretrained('nlpaueb/bert-base-greek-uncased-v1', do_lower_case = True, strip_accents = True)

config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Tokenization/ Sliding Window/ BIO Tagging

We will use the greekBERT tokenizer to tokenize the text. In the previous section we saw that a lot of documents have more than 512 tokens. So we will use a sliding window technique to convert the big documents to chunks. After we will tag the tokens with the mentions we have using the BIO tagging technique. The fact that we will create chunks from the documents creates a problem with the labels. There is not a single way to solve this , so we will experiment with this

In [12]:
def prepare_features(docs, y_labels, tokenizer, string_labels):
    all_input_ids = []
    all_attention_masks = []
    all_ner_labels = []     #  BIO tags
    all_class_labels = []   # multi-hot encoding
    all_patient_ids = []

    for doc_idx, doc in enumerate(docs):
        doc_y = y_labels[doc_idx]
        
        # 2. Tokenization & Sliding Window
        encoded = tokenizer(
            doc.text,
            padding='max_length',
            truncation=True,
            max_length=512,
            stride=64,
            return_overflowing_tokens=True,
            return_offsets_mapping=True
        )
        
        # for every chunk in the document
        for chunk_idx in range(len(encoded['input_ids'])):
            chunk_input_ids = encoded['input_ids'][chunk_idx]
            chunk_mask = encoded['attention_mask'][chunk_idx]
            chunk_offsets = encoded['offset_mapping'][chunk_idx]
            
            
            # Initialize BIO tags
            chunk_ner_tags = ['O'] * 512
            
            # Initialize the labels 
            chunk_class_label = np.zeros(110, dtype=int)
            
            # If the document has mention annotation for the labels
            if doc.mentions and len(doc.mentions) > 0:
                for i, (start, end) in enumerate(chunk_offsets):
                    
                    # Check if special token
                    if start==end==0:
                        chunk_ner_tags[i] = -100
                        continue
                    
                    for mention in doc.mentions:
                        if start >= mention.start and end <= mention.end:
                            if i == 0 or chunk_ner_tags[i-1] not in [f"B-{mention.code}", f"I-{mention.code}"]:
                                chunk_ner_tags[i] = f"B-{mention.code}"
                            else:
                                chunk_ner_tags[i] = f"I-{mention.code}"
                            index = np.where(string_labels == mention.code)
                            chunk_class_label[index] = 1
                            break
                    pass 
            else:
                # If the document does not have mentions then we set all tags to -100 so pytorch will ignore them
                chunk_ner_tags = [-100] * 512
                
                chunk_class_label = doc_y
                pass
                
            # Save the chunks results to the lists
            all_patient_ids.append(doc.patient_id)
            all_input_ids.append(chunk_input_ids)
            all_attention_masks.append(chunk_mask)
            all_ner_labels.append(chunk_ner_tags)
            all_class_labels.append(chunk_class_label)

    return all_input_ids, all_attention_masks, all_ner_labels, all_class_labels, all_patient_ids
    

## Convert the data to tensors

Let's say that this section is where analysis and experiments stop and starts the 'real' nlp task. To summarize our previous work:

1. We have 3 sets. Training/Validation/Test.

2. We have 4 lists: input_ids, attention_masks, ner_labels, class_labels, for each set.

3. We used a sliding window to create tokens_sets of max 512 tokens and to keep all information, we didnt truncate but we splitted the documents to chunks

4. The chunks of documents with mentions will have the labels from the mentions for each chunk

5. The chunks of documents without mentions will inherit the labels from the global document (it is possible that this way add false information).

#### Convert BIO Tags to numbers

Create tag to id dictionary

In [13]:
tag2id_dict = {}
tag2id_dict['O'] = 0
counter = 0
for code in string_labels:
    counter+=1
    tag2id_dict['B-' + code] = counter
    counter+=1
    tag2id_dict['I-' + code] = counter


In [14]:
def convert_tag2id(ner_labels,tag2id_dict):
    id_list = []
    for i in range(len(ner_labels)):
        if ner_labels[i] != -100:
            id_list.append(tag2id_dict[ner_labels[i]])
        else:
            id_list.append(-100)
    return id_list

#### Create tensors

In [15]:
def create_tensors(x_docs,y,tokenizer,string_labels,tag2id_dict):
    input_ids, attention_masks, ner_labels, labels , patients_ids= prepare_features(x_docs,y,tokenizer,string_labels)
    for i in range(len(ner_labels)):
        ner_labels[i] = convert_tag2id(ner_labels[i],tag2id_dict)

    input_ids = np.array(input_ids)
    attention_masks = np.array(attention_masks)
    ner_labels = np.array(ner_labels)
    labels = np.array(labels)
    patients_ids = np.array(patients_ids)
    return torch.tensor(input_ids).long() , torch.tensor(attention_masks).long() , torch.tensor(ner_labels).long() , torch.tensor(labels, dtype=torch.float32) , torch.tensor(patients_ids, dtype=torch.int)

#### Create TensorDataset and DataLoader

## Model

We will use the GreekBERT from hugging face

In [18]:
class ElCardioModel(nn.Module):
    def __init__(self):
        super(ElCardioModel, self).__init__()

        # Greek Bert with no heads
        self.bert = AutoModel.from_pretrained("nlpaueb/bert-base-greek-uncased-v1")

        # Dropout for regularization
        self.dropout = nn.Dropout(p=0.1)

        # Classification Head
        self.classification_head = nn.Linear(768,110)

        # NER Head
        self.ner_head = nn.Linear(768,221)


    def forward(self, input_ids, attention_mask):
        
        bert_output = self.bert(input_ids,attention_mask)
        sequence_bert_output = bert_output.last_hidden_state 

        sequence_bert_output = self.dropout(sequence_bert_output)
        # classification head
        classification = self.classification_head(sequence_bert_output[:, 0, :])

        # NER head
        ner = self.ner_head(sequence_bert_output)

        return classification, ner

In [ ]:
class ElCardioModelEntityPooling(nn.Module):
    def __init__(self):
       super().__init__()

        # Greek Bert with no heads
        self.bert = AutoModel.from_pretrained("nlpaueb/bert-base-greek-uncased-v1")

        # Dropout for regularization
        self.dropout = nn.Dropout(p=0.1)

        # Classification Head
        self.classification_head = nn.Linear(768*2,110)

        # NER Head
        self.ner_head = nn.Linear(768,221)


    def forward(self, input_ids, attention_mask, ner_labels=None, use_teacher_forcing=False):
        
        bert_output = self.bert(input_ids,attention_mask)
        sequence_bert_output = bert_output.last_hidden_state 
        sequence_bert_output = self.dropout(sequence_bert_output)

        # Extract CLS Token
        cls_token = sequence_bert_output[:,0,:]

        # Calculate ner tags
        ner_logits = self.ner_head(sequence_bert_output)

        # If we want to use Teacher Frocing
        if use_teacher_forcing and ner_labels is not None:
            # Check if we have the ground truth labels
            has_valid_labels = (ner_labels != -100).any(dim=1) # Shape: [batch_size]
            
            # Create the ground truth mask
            gold_mask = (ner_labels > 0).float()
            
            # Create the mask from the ner head
            ner_preds = torch.argmax(ner_logits, dim=-1)
            pred_mask = (ner_preds > 0).float()
            
            # If the doc is mentioned then keep the ground truth labels, if not keep the ner head prediction
            has_valid_labels_expanded = has_valid_labels.unsqueeze(-1).float()
            
            mention_mask = (gold_mask * has_valid_labels_expanded) + (pred_mask * (1 - has_valid_labels_expanded))
            mention_mask = mention_mask.unsqueeze(-1)
            
        else:
            # Inference, we use the ner head preidction
            ner_preds = torch.argmax(ner_logits, dim=-1)
            mention_mask = (ner_preds > 0).float().unsqueeze(-1)

        # Use the mention mask to delete the noise
        masked_sequence = sequence_bert_output * mention_mask

        # Mean Pooling of mentions
        sum_embeddings = torch.sum(masked_sequence, dim=1)

        num_mentions = torch.clamp(torch.sum(mention_mask, dim=1), min = 1e-9)
        pooled_mentions = sum_embeddings/num_mentions

        # Concatenate and Classification
        final_representation = torch.cat([cls_token, pooled_mentions], dim=1)

        classification = self.classification_head(final_representation)
        
        return classification, ner_logits

### Load Models

In [20]:
model_maxrecall = ElCardioModel().to(device)
model_maxrecall.load_state_dict(torch.load('/kaggle/input/notebooks/dimitriospakakis/elcardio-max-recall/best_elcardio_model_final.pth'))

model_maxprecision =ElCardioModelEntityPooling().to(device)
model_maxprecision.load_state_dict(torch.load('/kaggle/input/notebooks/dimitriospakakis/elcardio-entity-maxprecision/best_elcardio_model_phase3_maxprecision.pth'))

pytorch_model.bin:   0%|          | 0.00/454M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/bert-base-greek-uncased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<All keys matched successfully>

## Final Answer

In [ ]:
BATCH_SIZE=8
print("\nLoading Final Test Set...")
test_file_path = "/kaggle/input/datasets/dimitriospakakis/final-test-dataset/test_set.jsonl" 

test_documents = []
with open(test_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        test_documents.append(ClinicalDocument(patient_id=data['patient_id'], text=data['text'], document_labels=[], mentions=[]))

dummy_y = np.zeros((len(test_documents), len(string_labels)))

ext_test_input_ids, ext_test_attention_masks, _, _, ext_test_patients_ids = create_tensors(
    x_docs=test_documents, y=dummy_y, tokenizer=tokenizer, string_labels=string_labels, tag2id_dict=tag2id_dict
)

ext_test_dataset = TensorDataset(ext_test_input_ids, ext_test_attention_masks, ext_test_patients_ids)
ext_test_loader = DataLoader(ext_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
best_maxprecision_thresholds = np.load('/kaggle/input/notebooks/dimitriospakakis/elcardio-entity-maxprecision/best_sniper_thresholds.npy')
recall_global_threshold = 0.6
maxprecision_classes_mask = (best_maxprecision_thresholds < 0.8)

In [43]:

model_maxrecall.eval()
model_maxprecision.eval()

final_predictions=[]
print("Inference in Test Set...")
with torch.no_grad():
    for batch in ext_test_loader:
         b_input_ids = batch[0].to(device)
         b_attn_mask = batch[1].to(device)
         b_patient_ids = batch[2]

          # --- Forward Pass Precision ---
        logits_precision, _ = model(b_input_ids, b_attn_mask, ner_labels=None, use_teacher_forcing=False)
        probs_precision = torch.sigmoid(logits_precision).cpu().numpy()[0]

        # --- Forward Pass Recall ---
        logits_recall, _ = model_max_recall(b_input_ids, b_attn_mask)
        probs_recall = torch.sigmoid(logits_recall).cpu().numpy()[0]

        preds_precision = (probs_precision > best_maxprecision_thresholds).astype(int)
        preds_recall = (probs_recall > recall_global_threshold).astype(int)
        
        final_preds = np.zeros_like(preds_sniper)
        
         # Masking
        final_preds[maxprecision_classes_mask] = preds_precision[maxprecision_classes_mask]
        final_preds[~maxprecision_classes_mask] = preds_recall[~maxprecision_classes_mask]

         # Find indices where array is "1"
        predicted_indices = np.where(final_preds == 1)[0]

        predicted_codes = [string_labels[idx] for idx in predicted_indices]

        final_predictions.append({ "patient_id": int(patient_id),
        "annotations": predicted_codes})

output_filename = "ensembling_cascade.jsonl"
with open(output_filename, 'w', encoding = 'utf-8') as f:
    for prediction in final_predictions:
        f.write(json.dumps(prediction, ensure_ascii=False)+'\n')

print(f"Finished. The file {output_filename}")


Loading Final Test Set...
Inference in Test Set...
The submision file submission_ensemble_weighted.jsonl is ready!
